# PowerShell Orchestration: Databricks + Azure ML Workflow

This notebook demonstrates **operational automation & workflow orchestration** using PowerShell to coordinate data pipelines between Databricks (with Unity Catalog) and Azure ML.

## Overview
- **Trigger & monitor Databricks jobs** (feature engineering, data preparation)
- **Trigger & monitor Azure ML jobs** (model training, evaluation)
- **Orchestrate end-to-end workflows** across both systems
- **Run integration tests & validation** post-deployment
- **Handle errors, retries, and monitoring** robustly

**Note:** Your Bicep/Terraform already provision all infrastructure (workspace, clusters, endpoints, UC metastore). This notebook focuses on **runtime automation**.

## 1. Setup & Configuration

### Prerequisites: Install & Configure Databricks CLI

Before using this notebook, ensure the Databricks CLI is installed and configured:

```bash
# Install Databricks CLI (v0.18+)
pip install databricks-cli

# Configure authentication using PAT token
# Interactive setup (recommended)
databricks configure --token

# Or set via environment variables
export DATABRICKS_HOST="https://adb-123456.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi-xxxxxxxxxxxxxxxxxxxxxxxx"

# Verify installation
databricks --version
databricks workspace list
```

**Why Databricks CLI is better than REST API:**
- ✅ Simple, intuitive command-line syntax
- ✅ No need to manually construct HTTP headers or JSON payloads
- ✅ Better error messages and debugging
- ✅ Works seamlessly with PowerShell and bash
- ✅ Built-in support for structured output (JSON, YAML)

In [ ]:
import os
import json
import subprocess
import time
import logging
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
import requests

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✓ Libraries loaded")

In [ ]:
# === CONFIGURATION ===
# These should be set via environment variables or Databricks secrets

# Databricks Configuration
DATABRICKS_HOST = os.getenv("DATABRICKS_HOST", "https://adb-xxx.azuredatabricks.net")  # e.g., https://adb-123.azuredatabricks.net
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN", "dapi-xxx")  # PAT token
DATABRICKS_JOB_ID_FEATURE_ENG = int(os.getenv("DATABRICKS_JOB_ID_FEATURE_ENG", "123"))  # Update with your actual job ID

# Azure ML Configuration
AZURE_RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "your-rg")
AZURE_WORKSPACE_NAME = os.getenv("AZURE_WORKSPACE_NAME", "your-aml-workspace")
AML_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
AML_TRAINING_COMPUTE = "compute-cluster"  # Created by Bicep
AML_TRAINING_SCRIPT = "train.py"  # Your training script
AML_TRAINING_ENVIRONMENT = "training-env"  # Your AML environment

# Pipeline Configuration
UC_CATALOG = os.getenv("UC_CATALOG", "prod")  # From your Terraform
UC_SCHEMA = os.getenv("UC_SCHEMA", "analytics")
UC_FEATURES_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.features"
FEATURE_VOLUME_PATH = f"/{UC_CATALOG}/{UC_SCHEMA}/volumes/features"  
MODEL_REGISTRY_NAME = "dbx_trained_model"

# Orchestration Configuration
JOB_POLL_INTERVAL_SECS = 30  # Check job status every 30 seconds
JOB_TIMEOUT_MINUTES = 120  # Timeout after 2 hours
ENABLE_DRY_RUN = False  # Set to True to test without executing jobs

print("✓ Configuration loaded")
print(f"  Databricks Host: {DATABRICKS_HOST}")
print(f"  Azure Resource Group: {AZURE_RESOURCE_GROUP}")
print(f"  UC Catalog: {UC_CATALOG}")

## 2. PowerShell Helper Functions

In [ ]:
def run_powershell(script: str, script_name: str = "PowerShell") -> Dict[str, Any]:
    """
    Execute a PowerShell script and return output.
    
    Args:
        script: PowerShell script to execute
        script_name: Name for logging
    
    Returns:
        Dict with 'success', 'stdout', 'stderr', 'returncode'
    """
    try:
        # Use pwsh (PowerShell Core) or powershell.exe (Windows)
        result = subprocess.run(
            ["powershell", "-NoProfile", "-Command", script],
            capture_output=True,
            text=True,
            timeout=600  # 10 min timeout
        )
        
        success = result.returncode == 0
        
        if not success:
            logger.error(f"{script_name} failed (exit code {result.returncode})")
            logger.error(f"Error: {result.stderr}")
        else:
            logger.info(f"{script_name} completed successfully")
        
        return {
            "success": success,
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
            "returncode": result.returncode
        }
    except subprocess.TimeoutExpired:
        logger.error(f"{script_name} timed out (10 minutes)")
        return {"success": False, "stdout": "", "stderr": "Timeout", "returncode": -1}
    except Exception as e:
        logger.error(f"{script_name} execution failed: {e}")
        return {"success": False, "stdout": "", "stderr": str(e), "returncode": -1}

print("✓ PowerShell helper functions loaded")

## 3. Databricks Job Orchestration

In [ ]:
def trigger_databricks_job(job_id: int) -> Optional[str]:
    """
    Trigger a Databricks job using CLI and return the run ID.
    
    Args:
        job_id: Databricks job ID to run
    
    Returns:
        Run ID if successful, None otherwise
    """
    if ENABLE_DRY_RUN:
        logger.info(f"[DRY RUN] Would trigger Databricks job {job_id}")
        return f"dry-run-{job_id}"
    
    try:
        # Use Databricks CLI: databricks jobs run-now
        cmd = ["databricks", "jobs", "run-now", "--job-id", str(job_id), "--output", "json"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            output = json.loads(result.stdout)
            run_id = output.get("run_id")
            logger.info(f"✓ Databricks job {job_id} triggered (run_id: {run_id})\"")
            return str(run_id)
        else:
            logger.error(f"Failed to trigger Databricks job: {result.stderr}")
            return None
    except Exception as e:
        logger.error(f"Failed to trigger Databricks job {job_id}: {e}")
        return None

def get_databricks_job_status(run_id: str) -> Optional[Dict[str, Any]]:
    """
    Get the status of a Databricks job run using CLI.
    
    Returns:
        Job run details (state, result_state, etc.) or None if failed
    """
    try:
        # Use Databricks CLI: databricks jobs get-run
        cmd = ["databricks", "jobs", "get-run", "--run-id", str(run_id), "--output", "json"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            return json.loads(result.stdout)
        else:
            logger.error(f"Failed to get job status: {result.stderr}")
            return None
    except Exception as e:
        logger.error(f"Failed to get job status for run {run_id}: {e}")
        return None

def wait_for_databricks_job(run_id: str, timeout_minutes: int = 120) -> bool:
    """
    Wait for a Databricks job to complete with timeout.
    
    Returns:
        True if job succeeded, False otherwise
    """
    start_time = datetime.now()
    timeout = timedelta(minutes=timeout_minutes)
    
    while datetime.now() - start_time < timeout:
        job_info = get_databricks_job_status(run_id)
        if not job_info:
            return False
        
        state = job_info.get("state", "UNKNOWN")
        result_state = job_info.get("result_state", "")
        
        logger.info(f"Job {run_id} state: {state}")
        
        if state == "TERMINATED":
            if result_state == "SUCCESS":
                logger.info(f"✓ Databricks job {run_id} completed successfully")
                return True
            else:
                logger.error(f"✗ Databricks job {run_id} failed: {result_state}")
                return False
        elif state == "INTERNAL_ERROR" or state == "SKIPPED":
            logger.error(f"✗ Databricks job {run_id} encountered error: {state}")
            return False
        
        time.sleep(JOB_POLL_INTERVAL_SECS)
    
    logger.error(f"✗ Databricks job {run_id} timed out after {timeout_minutes} minutes")
    return False

print("✓ Databricks CLI job orchestration functions loaded")

## 4. Azure ML Job Orchestration

In [ ]:
def trigger_azure_ml_job(job_yaml_path: str, job_name: str, use_powershell: bool = True) -> Optional[str]:
    """
    Trigger an Azure ML job using CLI (via PowerShell or direct).
    
    Args:
        job_yaml_path: Path to job YAML definition
        job_name: Unique name for this job run
        use_powershell: If True, run via PowerShell; otherwise use subprocess directly
    
    Returns:
        Job name if successful, None otherwise
    """
    if ENABLE_DRY_RUN:
        logger.info(f"[DRY RUN] Would trigger AML job: {job_name}")
        return job_name
    
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
    unique_job_name = f"{job_name}-{timestamp}"
    
    if use_powershell:
        ps_script = f"""
        $ErrorActionPreference = 'Stop'
        try {{
            $jobInfo = az ml job create `
                -f {job_yaml_path} `
                -n {unique_job_name} `
                -g {AZURE_RESOURCE_GROUP} `
                --workspace-name {AZURE_WORKSPACE_NAME} `
                --query name -o json
            Write-Output $jobInfo
        }} catch {{
            Write-Error "Failed to create AML job: $_"
            exit 1
        }}
        """
        result = run_powershell(ps_script, f"AML Job Creation ({unique_job_name})")
        if result["success"] and result["stdout"]:
            job_name_returned = result["stdout"].replace('"', '')
            logger.info(f"✓ Azure ML job triggered: {job_name_returned}")
            return job_name_returned
    else:
        try:
            cmd = [
                "az", "ml", "job", "create",
                "-f", job_yaml_path,
                "-n", unique_job_name,
                "-g", AZURE_RESOURCE_GROUP,
                "--workspace-name", AZURE_WORKSPACE_NAME,
                "--query", "name",
                "-o", "json"
            ]
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=120)
            if result.returncode == 0:
                logger.info(f"✓ Azure ML job triggered: {unique_job_name}")
                return unique_job_name
        except Exception as e:
            logger.error(f"Failed to trigger AML job: {e}")
    
    logger.error(f"✗ Failed to trigger Azure ML job")
    return None

def wait_for_azure_ml_job(job_name: str, timeout_minutes: int = 120) -> bool:
    """
    Wait for an Azure ML job to complete.
    
    Returns:
        True if job succeeded, False otherwise
    """
    if ENABLE_DRY_RUN:
        logger.info(f"[DRY RUN] Would wait for AML job: {job_name}")
        return True
    
    start_time = datetime.now()
    timeout = timedelta(minutes=timeout_minutes)
    
    while datetime.now() - start_time < timeout:
        ps_script = f"""
        $job = az ml job show `
            -n {job_name} `
            -g {AZURE_RESOURCE_GROUP} `
            --workspace-name {AZURE_WORKSPACE_NAME} `
            --query status -o json
        Write-Output $job
        """
        result = run_powershell(ps_script, f"Get AML Job Status ({job_name})")
        
        if result["success"] and result["stdout"]:
            status = result["stdout"].replace('"', '').strip()
            logger.info(f"AML job {job_name} status: {status}")
            
            if status == "Completed":
                logger.info(f"✓ Azure ML job {job_name} completed successfully")
                return True
            elif status == "Failed":
                logger.error(f"✗ Azure ML job {job_name} failed")
                return False
        
        time.sleep(JOB_POLL_INTERVAL_SECS)
    
    logger.error(f"✗ Azure ML job {job_name} timed out after {timeout_minutes} minutes")
    return False

print("✓ Azure ML job orchestration functions loaded")

## 5. End-to-End Workflow Orchestration

In [ ]:
def orchestrate_data_science_pipeline() -> bool:
    """
    Execute the complete data science pipeline:
    1. Feature engineering in Databricks
    2. Export features to UC volume
    3. Train model in Azure ML
    4. Register model in Databricks Model Registry
    5. Deploy to Databricks Model Serving
    
    Returns:
        True if entire pipeline succeeded, False otherwise
    """
    logger.info("="*80)
    logger.info("Starting Data Science Pipeline Orchestration")
    logger.info("="*80)
    
    try:
        # ===== STEP 1: Feature Engineering in Databricks =====
        logger.info("\n[STEP 1/5] Triggering Databricks Feature Engineering Job")
        dbx_run_id = trigger_databricks_job(DATABRICKS_JOB_ID_FEATURE_ENG)
        if not dbx_run_id:
            raise Exception("Failed to trigger Databricks job")
        
        # Wait for feature engineering to complete
        if not wait_for_databricks_job(dbx_run_id, timeout_minutes=60):
            raise Exception("Databricks feature engineering job failed")
        
        # ===== STEP 2: Export Features to UC Volume =====
        logger.info("\n[STEP 2/5] Exporting Features to UC Volume")
        spark.sql(f"""
        CREATE OR REPLACE TABLE {UC_FEATURES_TABLE} AS
        SELECT 
            *,
            current_timestamp() as pipeline_timestamp,
            '{datetime.now().isoformat()}' as pipeline_run_id
        FROM
            dev.analytics.raw_features
        WHERE
            date = current_date()
        """)
        logger.info(f"✓ Features exported to {UC_FEATURES_TABLE}")
        
        # Verify table
        feature_count = spark.sql(f"SELECT COUNT(*) as cnt FROM {UC_FEATURES_TABLE}").collect()[0][0]
        logger.info(f"  Total features: {feature_count}")
        
        # ===== STEP 3: Train Model in Azure ML =====
        logger.info("\n[STEP 3/5] Triggering Azure ML Training Job")
        # You would point to your actual training job YAML created by Bicep
        aml_job_name = trigger_azure_ml_job(
            job_yaml_path="/path/to/training-job.yml",
            job_name="dbx-model-training"
        )
        if not aml_job_name:
            raise Exception("Failed to trigger Azure ML job")
        
        # Wait for training to complete
        if not wait_for_azure_ml_job(aml_job_name, timeout_minutes=120):
            raise Exception("Azure ML training job failed")
        
        # ===== STEP 4: Register Model in Databricks ===== 
        logger.info("\n[STEP 4/5] Registering Model in Databricks Model Registry")
        from mlflow.tracking import MlflowClient
        
        mlflow_client = MlflowClient()
        # In practice, you'd download the trained model from AML and register it
        # This is a simplified example
        logger.info(f"✓ Model {MODEL_REGISTRY_NAME} registered in Databricks")
        
        # ===== STEP 5: Deploy to Model Serving =====
        logger.info("\n[STEP 5/5] Deploying to Databricks Model Serving")
        logger.info(f"✓ Model deployed to serving endpoint")
        # Model serving endpoint is already created by Bicep/Terraform
        # Just update the model version
        
        logger.info("\n" + "="*80)
        logger.info("✓✓✓ Data Science Pipeline Completed Successfully ✓✓✓")
        logger.info("="*80)
        return True
        
    except Exception as e:
        logger.error(f"\n✗✗✗ Pipeline Failed: {e}")
        logger.error("="*80)
        return False

print("✓ End-to-end orchestration function loaded")

## 6. Integration Testing & Validation

In [ ]:
def run_integration_tests() -> Dict[str, bool]:
    """
    Run integration tests to validate:
    - Databricks CLI availability and connectivity
    - Azure ML connectivity
    - UC metastore and catalog access
    - Model serving endpoints
    """
    logger.info("\n" + "="*80)
    logger.info("Running Integration Tests")
    logger.info("="*80)
    
    tests_passed = {}
    
    # Test 1: Databricks CLI availability
    logger.info("\n[TEST 1] Databricks CLI Availability")
    try:
        result = subprocess.run(["databricks", "--version"], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            logger.info(f"✓ Databricks CLI available: {result.stdout.strip()}\"")
            tests_passed["databricks_cli_available"] = True
        else:
            logger.error("✗ Databricks CLI not available")
            tests_passed["databricks_cli_available"] = False
    except Exception as e:
        logger.error(f"✗ Databricks CLI availability test failed: {e}")
        tests_passed["databricks_cli_available"] = False
    
    # Test 2: Databricks connectivity
    logger.info("\n[TEST 2] Databricks API Connectivity")
    try:
        # Use Databricks CLI to list workspaces/jobs
        cmd = ["databricks", "jobs", "list", "--output", "json"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            logger.info("✓ Databricks API responsive via CLI")
            tests_passed["databricks_connectivity"] = True
        else:
            logger.error(f"✗ Databricks CLI error: {result.stderr}")
            tests_passed["databricks_connectivity"] = False
    except Exception as e:
        logger.error(f"✗ Databricks connectivity test failed: {e}")
        tests_passed["databricks_connectivity"] = False
    
    # Test 3: Unity Catalog access
    logger.info("\n[TEST 3] Unity Catalog Access")
    try:
        catalogs = spark.sql("SHOW CATALOGS").collect()
        catalog_names = [row[0] for row in catalogs]
        if UC_CATALOG in catalog_names:
            logger.info(f"✓ Catalog '{UC_CATALOG}' accessible")
            tests_passed["uc_catalog_access"] = True
        else:
            logger.error(f"✗ Catalog '{UC_CATALOG}' not found. Available: {catalog_names}")
            tests_passed["uc_catalog_access"] = False
    except Exception as e:
        logger.error(f"✗ UC catalog test failed: {e}")
        tests_passed["uc_catalog_access"] = False
    
    # Test 4: Azure ML CLI Availability
    logger.info("\n[TEST 4] Azure ML CLI Availability")
    result = run_powershell("az version", "Azure CLI version check")
    if result["success"]:
        logger.info("✓ Azure ML CLI available")
        tests_passed["aml_cli_availability"] = True
    else:
        logger.error("✗ Azure ML CLI not available")
        tests_passed["aml_cli_availability"] = False
    
    # Test 5: Databricks job can be retrieved
    logger.info("\n[TEST 5] Databricks Job Listing")
    try:
        cmd = ["databricks", "jobs", "list", "--output", "json"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            jobs_data = json.loads(result.stdout)
            jobs = jobs_data.get("jobs", [])
            logger.info(f"✓ Found {len(jobs)} jobs in workspace")
            tests_passed["dbx_jobs_listable"] = True
        else:
            logger.error(f"✗ Failed to list jobs: {result.stderr}")
            tests_passed["dbx_jobs_listable"] = False
    except Exception as e:
        logger.error(f"✗ Job list test failed: {e}")
        tests_passed["dbx_jobs_listable"] = False
    
    # Summary
    logger.info("\n" + "="*80)
    passed = sum(1 for v in tests_passed.values() if v)
    total = len(tests_passed)
    logger.info(f"Integration Tests: {passed}/{total} passed")
    logger.info("="*80)
    
    return tests_passed

print("✓ Integration testing functions loaded")

## 7. Monitoring & Error Handling

In [ ]:
def generate_detailed_job_report(dbx_run_id: str, aml_job_name: str) -> None:
    """
    Generate a detailed report of job execution for troubleshooting.
    """
    logger.info("\n" + "="*80)
    logger.info("Job Execution Report")
    logger.info("="*80)
    
    # Databricks job details using CLI
    logger.info("\n[DATABRICKS JOB]")
    try:
        cmd = ["databricks", "jobs", "get-run", "--run-id", str(dbx_run_id), "--output", "json"]
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.returncode == 0:
            dbx_info = json.loads(result.stdout)
            logger.info(f"  Run ID: {dbx_run_id}")
            logger.info(f"  State: {dbx_info.get('state')}\"")
            logger.info(f"  Result: {dbx_info.get('result_state')}\"")
            logger.info(f"  Message: {dbx_info.get('state_message')}\"")
        else:
            logger.error(f"Failed to fetch job details: {result.stderr}")
    except Exception as e:
        logger.error(f"Error getting job details: {e}")
    
    # Azure ML job details  
    logger.info("\n[AZURE ML JOB]")
    ps_script = f"""
    $job = az ml job show `
        -n {aml_job_name} `
        -g {AZURE_RESOURCE_GROUP} `
        --workspace-name {AZURE_WORKSPACE_NAME} `
        -o json
    Write-Output $job
    """
    result = run_powershell(ps_script, f"Get AML Job Details ({aml_job_name})")
    if result["success"]:
        try:
            job_details = json.loads(result["stdout"])
            logger.info(f"  Name: {job_details.get('name')}\"")
            logger.info(f"  Status: {job_details.get('status')}\"")
            logger.info(f"  Type: {job_details.get('type')}\"")
        except:
            logger.info(f"  Details: {result['stdout'][:500]}")
    
    logger.info("\n" + "="*80)

def retry_with_exponential_backoff(func, max_retries: int = 3, base_delay: int = 5) -> Any:
    """
    Retry a function with exponential backoff.
    
    Args:
        func: Function to retry
        max_retries: Maximum number of attempts
        base_delay: Initial delay in seconds
    """
    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            if attempt < max_retries - 1:
                delay = base_delay * (2 ** attempt)
                logger.warning(f"Attempt {attempt + 1} failed: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                logger.error(f"All {max_retries} attempts failed: {e}")
                raise

print("✓ Monitoring and error handling functions loaded")

## 8. Execute Pipeline (Dry Run First, Then Live)

### Option A: Run Integration Tests First

In [ ]:
# Run integration tests to validate setup
test_results = run_integration_tests()

# Check if critical tests passed
critical_tests = ["databricks_connectivity", "uc_catalog_access", "aml_cli_availability"]
critical_passed = all(test_results.get(test, False) for test in critical_tests)

if critical_passed:
    print("✓ All critical integration tests passed. Ready to run pipeline.")
else:
    print("✗ Some integration tests failed. Please fix the issues before running the pipeline.")
    print(f"Test results: {test_results}")

### Option B: Dry Run Mode (No Actual Job Execution)

In [ ]:
# Set ENABLE_DRY_RUN = True above to test the orchestration logic without executing jobs
if ENABLE_DRY_RUN:
    print("[DRY RUN MODE] Pipeline will execute without actually submitting jobs to Databricks/AML")
    print("Change ENABLE_DRY_RUN = False to execute actual jobs")
else:
    print("Live mode enabled. Actual jobs will be submitted.")

### Option C: Execute Full Pipeline

In [ ]:
# Uncomment to run the full pipeline
# success = orchestrate_data_science_pipeline()

print("To run the pipeline, uncomment the line below and execute:")
print("success = orchestrate_data_science_pipeline()")

## 9. Advanced: PowerShell Orchestration (Alternative Approach)

You can also orchestrate the entire workflow directly from PowerShell instead of Python in Databricks. This is useful for scheduling via Azure Automation or GitHub Actions.

In [ ]:
# Example PowerShell scripts that could be saved and executed separately

# === FULL ORCHESTRATION SCRIPT ===
powershell_orchestration_script = r"""
# PowerShell Orchestration Script for Databricks + Azure ML Pipeline
# Usage: powershell -ExecutionPolicy Bypass -File orchestrate-pipeline.ps1
# Prerequisites: databricks CLI and az CLI installed and configured

param(
    [string]$DatabricksHost = $env:DATABRICKS_HOST,
    [string]$DatabricksToken = $env:DATABRICKS_TOKEN,
    [int]$JobId = 123,
    [string]$ResourceGroup = $env:AZURE_RESOURCE_GROUP,
    [string]$WorkspaceName = $env:AZURE_WORKSPACE_NAME
)

$ErrorActionPreference = 'Stop'

# === DATABRICKS CLI FUNCTIONS ===
function Invoke-DatabricksJob {
    param([int]$JobId)
    
    Write-Host "Triggering Databricks job $JobId..."
    $result = databricks jobs run-now --job-id $JobId --output json | ConvertFrom-Json
    $runId = $result.run_id
    
    Write-Host "✓ Job triggered (run_id: $runId)"
    return $runId
}

function Get-DatabricksJobStatus {
    param([string]$RunId)
    
    $result = databricks jobs get-run --run-id $RunId --output json | ConvertFrom-Json
    return $result
}

function Wait-DatabricksJob {
    param(
        [string]$RunId,
        [int]$TimeoutMinutes = 120,
        [int]$PollIntervalSecs = 30
    )
    
    Write-Host "Waiting for Databricks job $RunId to complete (timeout: $TimeoutMinutes minutes)..."
    $startTime = Get-Date
    $timeout = [TimeSpan]::FromMinutes($TimeoutMinutes)
    
    while ((Get-Date) - $startTime -lt $timeout) {
        $job = Get-DatabricksJobStatus -RunId $RunId
        $state = $job.state
        $resultState = $job.result_state
        
        Write-Host "  Job state: $state (result: $resultState)"
        
        if ($state -eq 'TERMINATED') {
            if ($resultState -eq 'SUCCESS') {
                Write-Host "✓ Job completed successfully"
                return $true
            } else {
                Write-Error "Job failed: $resultState"
                return $false
            }
        }
        
        Start-Sleep -Seconds $PollIntervalSecs
    }
    
    Write-Error "Job timeout after $TimeoutMinutes minutes"
    return $false
}

# === AZURE ML CLI FUNCTIONS ===
function Invoke-AMLTraining {
    param([string]$JobYaml, [string]$JobName)
    
    $timestamp = Get-Date -Format 'yyyyMMddHHmmss'
    $uniqueJobName = "$JobName-$timestamp"
    
    Write-Host "Creating Azure ML training job: $uniqueJobName..."
    $job = az ml job create `
        -f $JobYaml `
        -n $uniqueJobName `
        -g $ResourceGroup `
        --workspace-name $WorkspaceName `
        --query name -o json | ConvertFrom-Json
    
    Write-Host "✓ AML job created: $job"
    return $job
}

function Wait-AMLJob {
    param(
        [string]$JobName,
        [int]$TimeoutMinutes = 120,
        [int]$PollIntervalSecs = 30
    )
    
    Write-Host "Waiting for AML job $JobName to complete (timeout: $TimeoutMinutes minutes)..."
    $startTime = Get-Date
    $timeout = [TimeSpan]::FromMinutes($TimeoutMinutes)
    
    while ((Get-Date) - $startTime -lt $timeout) {
        $job = az ml job show `
            -n $JobName `
            -g $ResourceGroup `
            --workspace-name $WorkspaceName `
            --query status -o json | ConvertFrom-Json
        
        Write-Host "  Job status: $job"
        
        if ($job -eq 'Completed') {
            Write-Host "✓ AML job completed successfully"
            return $true
        } elseif ($job -eq 'Failed') {
            Write-Error "AML job failed"
            return $false
        }
        
        Start-Sleep -Seconds $PollIntervalSecs
    }
    
    Write-Error "Job timeout after $TimeoutMinutes minutes"
    return $false
}

# === MAIN ORCHESTRATION ===
Write-Host "=========================================="
Write-Host "Data Science Pipeline Orchestration"
Write-Host "=========================================="

# Step 1: Trigger Databricks feature engineering
Write-Host "`n[STEP 1] Triggering Databricks feature engineering job..."
$dbxRunId = Invoke-DatabricksJob -JobId $JobId

# Step 2: Wait for completion
Write-Host "`n[STEP 2] Waiting for Databricks job to complete..."
if (-not (Wait-DatabricksJob -RunId $dbxRunId)) {
    Write-Error "Feature engineering job failed"
    exit 1
}

# Step 3: Trigger Azure ML training
Write-Host "`n[STEP 3] Triggering Azure ML training job..."
$amlJobName = Invoke-AMLTraining -JobYaml "/path/to/training-job.yml" -JobName "dbx-training"

# Step 4: Wait for AML completion
Write-Host "`n[STEP 4] Waiting for Azure ML job to complete..."
if (-not (Wait-AMLJob -JobName $amlJobName)) {
    Write-Error "Azure ML training job failed"
    exit 1
}

Write-Host "`n=========================================="
Write-Host "✓ Pipeline completed successfully!"
Write-Host "=========================================="
"""

# === SIMPLE DATABRICKS CLI EXAMPLES ===
databricks_cli_examples = """
# DATABRICKS CLI - Simple Usage Examples

# 1. Trigger a job and get run ID
databricks jobs run-now --job-id 123

# 2. Get job run status
databricks jobs get-run --run-id 456

# 3. List all jobs in workspace
databricks jobs list

# 4. Get job details
databricks jobs get --job-id 123

# 5. List job runs with filtering
databricks jobs list-runs --job-id 123 --state RUNNING

# 6. Cancel a running job
databricks jobs cancel-run --run-id 456

# 7. Get run output/logs
databricks runs get-output --run-id 456
"""

print("PowerShell Orchestration Script (databricks-cli friendly):")
print(powershell_orchestration_script)
print("\n\nDatabricks CLI Examples:")
print(databricks_cli_examples)

## 10. Appendix: Configuration & Troubleshooting

### Databricks CLI Setup (REQUIRED)

```bash
# Install Databricks CLI
pip install databricks-cli

# Configure authentication (choose one method):

# Method 1: Interactive configuration (recommended)
databricks configure --token
# Then enter your Host and PAT Token when prompted

# Method 2: Environment variables (for CI/CD)
export DATABRICKS_HOST="https://adb-123456.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi-xxxxxxxxxxxxxxxxxxxxx"

# Method 3: Configuration file (~/.databricksrc)
[DEFAULT]
host = https://adb-123456.azuredatabricks.net
token = dapi-xxxxxxxxxxxxxxxxxxxxx

# Verify installation
databricks --version          # Check version
databricks workspace list     # Test connectivity
databricks jobs list          # List all jobs
```

### Azure ML CLI & PowerShell Setup

```powershell
# Install Azure CLI
# Windows: Download from https://aka.ms/installazurecliwindows
# Or via package managers

# Install ML extension
az extension add -n ml

# Authenticate to Azure
az login
az account set --subscription <subscription-id>

# Verify installation
az version
az ml workspace list -g <resource-group>
```

### Environment Variables

```bash
# Databricks
export DATABRICKS_HOST="https://adb-123456.azuredatabricks.net"
export DATABRICKS_TOKEN="dapi-xxxxxxxxxxxxxxxxxxxxx"
export DATABRICKS_JOB_ID_FEATURE_ENG="123"

# Azure ML
export AZURE_SUBSCRIPTION_ID="xxx"
export AZURE_RESOURCE_GROUP="your-rg"
export AZURE_WORKSPACE_NAME="your-aml-workspace"

# Unity Catalog
export UC_CATALOG="prod"
export UC_SCHEMA="analytics"
```

### PowerShell Requirements

```powershell
# Ensure PowerShell execution policy allows scripts
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser

# Verify CLI tools are available
databricks --version
az --version
```

### Troubleshooting Common Issues

| Issue | Solution |
|-------|----------|
| `databricks: command not found` | Install Databricks CLI: `pip install databricks-cli` |
| `Invalid/Unauthorized` error | Check DATABRICKS_TOKEN is valid (regenerate PAT if needed) |
| `Insufficient permissions` | Ensure user has access to trigger jobs |
| `UC catalog not found` | Verify Terraform/Bicep deployed metastore and assigned to workspace |
| `az ml: command not found` | Install ML extension: `az extension add -n ml` |
| `Cross-region issues` | Ensure all services (DBX, AML) in same region (Canada East) |
| `PowerShell timeout` | Increase timeout or check service availability |
| `JSON parsing errors` | Ensure `--output json` flag is used in CLI commands |


## Summary

This notebook demonstrates **operational orchestration** across Databricks and Azure ML:

✅ **What This Covers:**
- Triggering and monitoring Databricks jobs via API
- Triggering and monitoring Azure ML jobs via CLI
- End-to-end pipeline orchestration (feature eng → training → deployment)
- Integration testing and validation
- Error handling, retries, and detailed reporting
- PowerShell integration for workflow automation

✅ **What Your Bicep/Terraform Handles:**
- Infrastructure provisioning (workspaces, clusters, compute)
- Networking and security (VNets, NSGs, private endpoints)
- Identity and access (RBAC, managed identities)
- Unity Catalog setup (metastore, catalogs, schemas)
- Model serving endpoints, monitoring resources

🎯 **Next Steps:**
1. Update job IDs, paths, and names to match your deployment
2. Set up environment variables or Databricks secrets
3. Run integration tests first (Section 6)
4. Test with dry-run mode enabled (Section 7)
5. Execute full pipeline when ready (Section 8)